In [1]:
# Verify GPU access on Colab
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

Thu Jul 16 13:28:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   48C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
#Verify PyTorch sees the GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA L4
GPU memory: 23.66 GB


In [3]:
#Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
#Creating project folders on Drive
import os

DRIVE_PROJECT = '/content/drive/MyDrive/msc_deepfake_project'
folders_to_create = [
    f'{DRIVE_PROJECT}/videos/generated/ltx',
    f'{DRIVE_PROJECT}/videos/generated/hunyuan',
    f'{DRIVE_PROJECT}/videos/generated/wan',
    f'{DRIVE_PROJECT}/videos/curated',
    f'{DRIVE_PROJECT}/metadata',
    f'{DRIVE_PROJECT}/mllm_outputs',
    f'{DRIVE_PROJECT}/logs',
]

for folder in folders_to_create:
    os.makedirs(folder, exist_ok=True)
    print(f"Ready: {folder}")

print("\nAll project folders created on Google Drive.")

Ready: /content/drive/MyDrive/msc_deepfake_project/videos/generated/ltx
Ready: /content/drive/MyDrive/msc_deepfake_project/videos/generated/hunyuan
Ready: /content/drive/MyDrive/msc_deepfake_project/videos/generated/wan
Ready: /content/drive/MyDrive/msc_deepfake_project/videos/curated
Ready: /content/drive/MyDrive/msc_deepfake_project/metadata
Ready: /content/drive/MyDrive/msc_deepfake_project/mllm_outputs
Ready: /content/drive/MyDrive/msc_deepfake_project/logs

All project folders created on Google Drive.


In [5]:
#Installing all dependencies for LTX-Video
!pip install -q --upgrade diffusers transformers accelerate imageio imageio-ffmpeg sentencepiece
!pip install -q einops
print("Installation complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 113.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 141.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 79.1 MB/s eta 0:00:00
Installation complete.


In [6]:
#Verifying the installed versions
import diffusers, transformers, torch
print(f"diffusers: {diffusers.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"torch: {torch.__version__}")

diffusers: 0.39.0
transformers: 5.14.1
torch: 2.11.0+cu128


In [7]:
#Loading the LTX-Video on L4 (simpler settings, more memory available)
from diffusers import LTXPipeline
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

print("Loading LTX-Video model...")

pipe = LTXPipeline.from_pretrained(
    "Lightricks/LTX-Video",
    torch_dtype=torch.bfloat16
)

pipe.to("cuda")

print("LTX-Video model loaded successfully on L4.")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"GPU memory available: {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading LTX-Video model...


model_index.json:   0%|          | 0.00/412 [00:00<?, ?B/s]

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

LTX-Video model loaded successfully on L4.
GPU memory used: 14.25 GB
GPU memory available: 23.66 GB


In [8]:
#Generating first video on L4
import torch
from diffusers.utils import export_to_video
from datetime import datetime

prompt = "A woman with long brown hair smiling gently at the camera in a sunlit park, natural lighting, soft breeze moving her hair, cinematic quality"
negative_prompt = "blurry, low quality, distorted, unnatural"

generator = torch.Generator(device="cuda").manual_seed(42)

print(f"Prompt: {prompt}\n")
print("Generating video on L4 (should take 3-7 minutes)...")

start_time = datetime.now()

video_frames = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    width=704,
    height=480,
    num_frames=73,
    num_inference_steps=30,
    guidance_scale=3.0,
    generator=generator
).frames[0]

end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

print(f"\nGeneration complete in {duration:.1f} seconds ({duration/60:.1f} minutes)")
print(f"Number of frames: {len(video_frames)}")

Prompt: A woman with long brown hair smiling gently at the camera in a sunlit park, natural lighting, soft breeze moving her hair, cinematic quality

Generating video on L4 (should take 3-7 minutes)...


  0%|          | 0/30 [00:00<?, ?it/s]


Generation complete in 27.7 seconds (0.5 minutes)
Number of frames: 73


In [9]:
#Saving video and metadata to Drive
from datetime import datetime
import json

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
video_id = f"ltx_001_{timestamp}"

video_path = f'{DRIVE_PROJECT}/videos/generated/ltx/{video_id}.mp4'
metadata_path = f'{DRIVE_PROJECT}/metadata/{video_id}.json'

# Export the video
export_to_video(video_frames, video_path, fps=24)

# Save metadata
metadata = {
    "video_id": video_id,
    "generator": "LTX-Video",
    "generator_version": "Lightricks/LTX-Video",
    "prompt": prompt,
    "negative_prompt": negative_prompt,
    "seed": 42,
    "width": 512,
    "height": 320,
    "num_frames": 49,
    "fps": 24,
    "num_inference_steps": 30,
    "guidance_scale": 3.0,
    "generation_time_seconds": duration,
    "generation_datetime": timestamp,
    "gpu_used": "T4",
    "optimizations": ["cpu_offload", "vae_slicing", "vae_tiling"]
}

with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Video saved: {video_path}")
print(f"Metadata saved: {metadata_path}")

Video saved: /content/drive/MyDrive/msc_deepfake_project/videos/generated/ltx/ltx_001_20260716_133303.mp4
Metadata saved: /content/drive/MyDrive/msc_deepfake_project/metadata/ltx_001_20260716_133303.json


In [10]:
#View the generated video inline
from IPython.display import Video
Video(video_path, embed=True, width=400)

In [11]:
#Downloading the video to my local machine
from google.colab import files
files.download(video_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>